In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install librosa noisereduce pydub tensorflow

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
print(tf.__version__)

2.20.0


In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
respiratory_path = "/content/drive/MyDrive/AICA_Health_Project/datasets/respiratory_sound_database/Respiratory_Sound_Database/audio_and_txt_files"

lung_path = "/content/drive/MyDrive/AICA_Health_Project/datasets/Audio Files"

cremad_path = "/content/drive/MyDrive/AICA_Health_Project/datasets/AudioWAV"

In [ ]:
print(os.listdir(respiratory_path)[:5])

print(os.listdir(lung_path)[:5])

print(os.listdir(cremad_path)[:5])

['160_1b3_Tc_mc_AKGC417L.wav', '160_1b3_Al_mc_AKGC417L.wav', '160_1b3_Pr_mc_AKGC417L.txt', '160_1b3_Lr_mc_AKGC417L.wav', '160_1b4_Al_mc_AKGC417L.wav']
['BP103_N,N,P R U,81,F.wav', 'BP109_N,N,P L M,26,M.wav', 'BP105_Lung Fibrosis,Crep,A U R,44,M.wav', 'BP108_COPD,E W,P R L ,63,M.wav', 'BP104_Asthma,E W,P L U,45,F.wav']
['1080_DFA_DIS_XX.wav', '1079_TIE_SAD_XX.wav', '1079_TIE_NEU_XX.wav', '1079_TSI_NEU_XX.wav', '1079_TIE_FEA_XX.wav']


In [ ]:
def extract_features(file_path):

    try:
        audio, sr = librosa.load(file_path, sr=None)

        # Skip empty audio
        if len(audio) == 0:
            return None

        # Avoid divide by zero
        max_value = np.max(np.abs(audio))

        if max_value == 0:
            return None

        # Normalize
        audio = audio / max_value

        # Silence Removal
        trimmed_audio, _ = librosa.effects.trim(audio)

        # Skip if audio becomes empty
        if len(trimmed_audio) < 2048:
            return None

        # MFCC Extraction
        mfcc = librosa.feature.mfcc(
            y=trimmed_audio,
            sr=sr,
            n_mfcc=40
        )

        mfcc_mean = mfcc.mean(axis=1)

        return mfcc_mean

    except Exception as e:
        print("Error:", file_path)

        return None

In [ ]:
# Process Respiratory Dataset
features = []
labels = []

for file in os.listdir(respiratory_path):

    if file.endswith(".wav"):

        file_path = os.path.join(respiratory_path, file)

        feature = extract_features(file_path)

        if feature is not None:

            features.append(feature)

            labels.append("respiratory")

In [ ]:
#Process Lung Dataset
for file in os.listdir(lung_path):

    if file.endswith(".wav"):

        file_path = os.path.join(lung_path, file)

        feature = extract_features(file_path)

        if feature is not None:

            features.append(feature)

            labels.append("lung")

In [ ]:
# Process CREMAD Dataset
for file in os.listdir(cremad_path):

    if file.endswith(".wav"):

        file_path = os.path.join(cremad_path, file)

        feature = extract_features(file_path)

        if feature is not None:

            features.append(feature)

            labels.append("emotion")

In [ ]:
#Create Final Dataset
df = pd.DataFrame(
    features,
    columns=[f"MFCC_{i}" for i in range(40)]
)

df["label"] = labels

df


,MFCC_0,MFCC_1,MFCC_2,MFCC_3,MFCC_4,MFCC_5,MFCC_6,MFCC_7,MFCC_8,MFCC_9,MFCC_10,MFCC_11,MFCC_12,MFCC_13,MFCC_14,MFCC_15,MFCC_16,MFCC_17,MFCC_18,MFCC_19,MFCC_20,MFCC_21,MFCC_22,MFCC_23,MFCC_24,MFCC_25,MFCC_26,MFCC_27,MFCC_28,MFCC_29,MFCC_30,MFCC_31,MFCC_32,MFCC_33,MFCC_34,MFCC_35,MFCC_36,MFCC_37,MFCC_38,MFCC_39,label
0,-314.533295,138.169693,67.244629,25.110495,22.168137,27.873199,25.767147,17.740784,10.675731,8.526386,9.137811,11.636518,11.171385,10.146179,6.018970,3.432355,1.993294,3.322440,3.581888,3.621322,3.630998,2.592503,1.807876,1.306239,2.870646,2.560257,2.865289,2.048939,2.457773,2.047640,3.175553,3.770144,3.580523,2.663612,1.308643,1.094068,1.397662,3.122508,2.653219,1.918069,respiratory
1,-378.353699,102.738274,64.840103,34.942257,22.513594,20.320244,20.775517,20.913235,19.649258,17.329294,15.094975,13.206498,11.356959,9.637511,8.216071,7.074880,6.350301,6.209100,6.463207,6.657328,6.422843,5.705823,4.706558,3.672120,2.821566,2.362520,2.312631,2.382266,2.308674,2.193096,2.228919,2.254297,1.975007,1.458922,1.047737,0.902080,0.955231,1.086238,1.170348,1.160116,respiratory
2,-377.652252,84.995285,71.308640,55.785221,42.125782,31.562075,24.012589,19.052938,15.999642,14.125464,12.832743,11.728978,10.597485,9.384645,8.188562,7.192468,6.535255,6.211318,6.047701,5.827382,5.456811,5.002329,4.562236,4.153697,3.732276,3.299957,2.928169,2.670539,2.521125,2.435449,2.323471,2.089123,1.749614,1.421958,1.169738,0.969860,0.805071,0.674184,0.579796,0.541791,respiratory
3,-373.978699,96.170074,65.512604,38.531643,24.864338,21.427256,21.179132,20.074961,17.730923,15.430109,14.058709,13.170322,11.795962,9.881367,8.200629,7.201882,6.673080,6.323149,6.141785,6.198695,6.329175,6.130277,5.312333,4.020422,2.811346,2.257904,2.394064,2.639875,2.479830,2.043691,1.807833,1.903791,1.980350,1.738249,1.330239,1.130880,1.253453,1.407350,1.269495,0.917983,respiratory
4,-374.564392,93.873856,66.821350,41.049084,26.642260,22.715921,22.711765,22.093586,20.383135,18.699614,17.298405,15.507779,12.956900,10.186257,8.106308,7.205027,7.303972,7.806870,8.068578,7.733472,6.897661,5.941336,5.173467,4.645619,4.248096,3.878615,3.488075,3.067027,2.683619,2.500515,2.640332,2.966691,3.063184,2.590183,1.685756,0.884280,0.597829,0.759080,1.002638,1.107330,respiratory
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8692,-164.891373,107.691383,45.994995,26.754650,12.380617,7.448959,1.967861,-3.921453,2.213546,-2.921345,-5.818723,-0.843532,-3.249035,-3.963472,-4.138944,-3.746255,-3.789826,-5.342563,-3.814027,-3.012420,-3.075929,4.551746,2.042935,2.658414,3.001451,-1.545643,-1.608092,3.112715,6.033406,5.891118,1.260730,0.379450,1.884516,1.546890,1.535191,-0.723229,-1.179331,-2.664740,-0.529202,2.381469,emotion
8693,-186.020615,112.033356,35.789379,18.760258,8.866562,6.789227,0.624307,-12.497063,2.664590,2.037013,-5.500131,-3.693132,-3.890115,-1.409668,-4.111963,-4.458514,-4.623692,-3.668110,-3.778800,-4.948789,-2.327234,1.428719,1.310393,2.866991,-1.138644,-0.626829,-0.719004,-1.880446,0.753261,5.191069,0.737146,0.797573,0.209986,0.837191,3.019883,3.631299,2.779552,0.189302,0.935070,2.147069,emotion
8694,-112.231071,99.433533,55.713417,33.616936,19.264114,1.595665,-0.712457,-0.936219,-0.126132,-2.435170,-3.608038,0.285069,-1.292433,-1.718323,-3.359527,-4.625218,-4.522385,-6.396152,-5.167612,-1.774554,-1.515271,0.329019,0.819395,2.869414,1.831589,0.184607,-0.582183,1.770158,3.721215,3.912431,3.826763,0.769668,1.023580,1.635620,0.775310,-0.103709,0.063775,-2.222892,-1.117260,0.085571,emotion
8695,-147.592392,85.866722,47.205723,38.969967,14.959960,5.442458,4.453246,1.157580,1.997117,-5.801095,-7.878387,-0.390868,-5.354165,-3.314533,-2.708359,-5.465576,-6.122663,-6.204161,-4.734056,-0.445316,-2.771740,-0.456298,1.252466,1.029614,0.429969,-0.325042,0.263024,1.567164,5.888305,5.951907,3.464818,2.773726,2.606628,1.559631,3.114499,0.668244,0.189054,-0.927283,-2.8

In [ ]:
df.shape

(8697, 41)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8697 entries, 0 to 8696
Data columns (total 41 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   MFCC_0   8697 non-null   float32
 1   MFCC_1   8697 non-null   float32
 2   MFCC_2   8697 non-null   float32
 3   MFCC_3   8697 non-null   float32
 4   MFCC_4   8697 non-null   float32
 5   MFCC_5   8697 non-null   float32
 6   MFCC_6   8697 non-null   float32
 7   MFCC_7   8697 non-null   float32
 8   MFCC_8   8697 non-null   float32
 9   MFCC_9   8697 non-null   float32
 10  MFCC_10  8697 non-null   float32
 11  MFCC_11  8697 non-null   float32
 12  MFCC_12  8697 non-null   float32
 13  MFCC_13  8697 non-null   float32
 14  MFCC_14  8697 non-null   float32
 15  MFCC_15  8697 non-null   float32
 16  MFCC_16  8697 non-null   float32
 17  MFCC_17  8697 non-null   float32
 18  MFCC_18  8697 non-null   float32
 19  MFCC_19  8697 non-null   float32
 20  MFCC_20  8697 non-null   float32
 21  MFCC_21  8697 

In [ ]:
# Label Encoding
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(df["label"])

X = df.drop("label", axis=1)

In [ ]:
y

array([2, 2, 2, ..., 0, 0, 0])

In [ ]:
X

,MFCC_0,MFCC_1,MFCC_2,MFCC_3,MFCC_4,MFCC_5,MFCC_6,MFCC_7,MFCC_8,MFCC_9,MFCC_10,MFCC_11,MFCC_12,MFCC_13,MFCC_14,MFCC_15,MFCC_16,MFCC_17,MFCC_18,MFCC_19,MFCC_20,MFCC_21,MFCC_22,MFCC_23,MFCC_24,MFCC_25,MFCC_26,MFCC_27,MFCC_28,MFCC_29,MFCC_30,MFCC_31,MFCC_32,MFCC_33,MFCC_34,MFCC_35,MFCC_36,MFCC_37,MFCC_38,MFCC_39
0,-314.533295,138.169693,67.244629,25.110495,22.168137,27.873199,25.767147,17.740784,10.675731,8.526386,9.137811,11.636518,11.171385,10.146179,6.018970,3.432355,1.993294,3.322440,3.581888,3.621322,3.630998,2.592503,1.807876,1.306239,2.870646,2.560257,2.865289,2.048939,2.457773,2.047640,3.175553,3.770144,3.580523,2.663612,1.308643,1.094068,1.397662,3.122508,2.653219,1.918069
1,-378.353699,102.738274,64.840103,34.942257,22.513594,20.320244,20.775517,20.913235,19.649258,17.329294,15.094975,13.206498,11.356959,9.637511,8.216071,7.074880,6.350301,6.209100,6.463207,6.657328,6.422843,5.705823,4.706558,3.672120,2.821566,2.362520,2.312631,2.382266,2.308674,2.193096,2.228919,2.254297,1.975007,1.458922,1.047737,0.902080,0.955231,1.086238,1.170348,1.160116
2,-377.652252,84.995285,71.308640,55.785221,42.125782,31.562075,24.012589,19.052938,15.999642,14.125464,12.832743,11.728978,10.597485,9.384645,8.188562,7.192468,6.535255,6.211318,6.047701,5.827382,5.456811,5.002329,4.562236,4.153697,3.732276,3.299957,2.928169,2.670539,2.521125,2.435449,2.323471,2.089123,1.749614,1.421958,1.169738,0.969860,0.805071,0.674184,0.579796,0.541791
3,-373.978699,96.170074,65.512604,38.531643,24.864338,21.427256,21.179132,20.074961,17.730923,15.430109,14.058709,13.170322,11.795962,9.881367,8.200629,7.201882,6.673080,6.323149,6.141785,6.198695,6.329175,6.130277,5.312333,4.020422,2.811346,2.257904,2.394064,2.639875,2.479830,2.043691,1.807833,1.903791,1.980350,1.738249,1.330239,1.130880,1.253453,1.407350,1.269495,0.917983
4,-374.564392,93.873856,66.821350,41.049084,26.642260,22.715921,22.711765,22.093586,20.383135,18.699614,17.298405,15.507779,12.956900,10.186257,8.106308,7.205027,7.303972,7.806870,8.068578,7.733472,6.897661,5.941336,5.173467,4.645619,4.248096,3.878615,3.488075,3.067027,2.683619,2.500515,2.640332,2.966691,3.063184,2.590183,1.685756,0.884280,0.597829,0.759080,1.002638,1.107330
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8692,-164.891373,107.691383,45.994995,26.754650,12.380617,7.448959,1.967861,-3.921453,2.213546,-2.921345,-5.818723,-0.843532,-3.249035,-3.963472,-4.138944,-3.746255,-3.789826,-5.342563,-3.814027,-3.012420,-3.075929,4.551746,2.042935,2.658414,3.001451,-1.545643,-1.608092,3.112715,6.033406,5.891118,1.260730,0.379450,1.884516,1.546890,1.535191,-0.723229,-1.179331,-2.664740,-0.529202,2.381469
8693,-186.020615,112.033356,35.789379,18.760258,8.866562,6.789227,0.624307,-12.497063,2.664590,2.037013,-5.500131,-3.693132,-3.890115,-1.409668,-4.111963,-4.458514,-4.623692,-3.668110,-3.778800,-4.948789,-2.327234,1.428719,1.310393,2.866991,-1.138644,-0.626829,-0.719004,-1.880446,0.753261,5.191069,0.737146,0.797573,0.209986,0.837191,3.019883,3.631299,2.779552,0.189302,0.935070,2.147069
8694,-112.231071,99.433533,55.713417,33.616936,19.264114,1.595665,-0.712457,-0.936219,-0.126132,-2.435170,-3.608038,0.285069,-1.292433,-1.718323,-3.359527,-4.625218,-4.522385,-6.396152,-5.167612,-1.774554,-1.515271,0.329019,0.819395,2.869414,1.831589,0.184607,-0.582183,1.770158,3.721215,3.912431,3.826763,0.769668,1.023580,1.635620,0.775310,-0.103709,0.063775,-2.222892,-1.117260,0.085571
8695,-147.592392,85.866722,47.205723,38.969967,14.959960,5.442458,4.453246,1.157580,1.997117,-5.801095,-7.878387,-0.390868,-5.354165,-3.314533,-2.708359,-5.465576,-6.122663,-6.204161,-4.734056,-0.445316,-2.771740,-0.456298,1.252466,1.029614,0.429969,-0.325042,0.263024,1.567164,5.888305,5.951907,3.464818,2.773726,2.606628,1.559631,3.114499,0.668244,0.189054,-0.927283,-2.822653,-0.201877


In [ ]:
print(encoder.classes_)

['emotion' 'lung' 'respiratory']


In [ ]:
# Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Build Deep Learning Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(128, activation='relu', input_shape=(40,)),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9989 - loss: 0.0032 - val_accuracy: 0.9971 - val_loss: 0.0096
Epoch 2/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9993 - loss: 0.0022 - val_accuracy: 0.9978 - val_loss: 0.0116
Epoch 3/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9991 - loss: 0.0026 - val_accuracy: 0.9978 - val_loss: 0.0095
Epoch 4/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9991 - loss: 0.0018 - val_accuracy: 0.9978 - val_loss: 0.0141
Epoch 5/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9989 - loss: 0.0023 - val_accuracy: 0.9978 - val_loss: 0.0099
Epoch 6/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9991 - loss: 0.0038 - val_accuracy: 0.9964 - val_loss: 0.0085
Epoch 7/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9991 - loss: 0.0025 - val_accuracy: 0.9978 - val_loss: 0.0101
Epoch 8/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9993 - loss: 0.0017 - val_accuracy: 0.

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9960 - loss: 0.0279
Accuracy: 0.995976984500885
